# Step 6: Tracing one user through the whole pipeline

So far we've trusted a single aggregate number — RMSE across tens of
thousands of hidden ratings. That's the right metric to optimize, but
it can feel abstract. This notebook does the opposite: pick ONE real
user, and look at exactly what happened to their ratings at every
stage.

**The three views of the same user:**
1. **Full history** — every rating they ever made, in `train.csv`
2. **Visible** — Has Ratings. It is the subset of those ratings the model was ALLOWED to learn from
3. **Hidden** — Has Ratings but we hide it during training. It is the subset we held back; the model never saw these during training

Then we check: for their hidden ratings (which the model never saw),
how close did the model's predictions come to what they actually rated?

This is the same hide-and-predict idea as before, just zoomed in on a
single person instead of averaged across everyone.

In [1]:
# This code is part of a movie recommendation system that uses the SVD++ algorithm to predict user ratings for movies. The code performs the following steps:
# 1. It loads the training data and selects a random subset of users to keep for training the model.
# 2. It splits the training data into a visible set (used for training) and a hidden set (used for evaluation).
# 3. It trains the SVD++ model on the visible data and measures the training time.
# 4. It predicts the ratings for the hidden set using the trained model and measures the prediction time.
# 5. Finally, it computes the Root Mean Squared Error (RMSE) between the actual ratings and the predicted ratings for the hidden set to evaluate the model's performance.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from surprise import SVDpp, Dataset, Reader

DATA_DIR = "data"
N_USERS_TO_KEEP = 15_000

train_full = pd.read_csv(f"{DATA_DIR}/train.csv")
rng = np.random.default_rng(42)
all_users = train_full["userId"].unique()
n_to_keep = min(N_USERS_TO_KEEP, len(all_users))
chosen_users = rng.choice(all_users, size=n_to_keep, replace=False)

train = train_full[train_full["userId"].isin(chosen_users)].reset_index(drop=True)
visible, hidden = train_test_split(train, test_size=0.2, random_state=42)

print(f"Visible: {len(visible):,}   Hidden: {len(hidden):,}")

Visible: 744,161   Hidden: 186,041


## Train the model (best config from tuning)

Using the winning settings from Step 5 — the original defaults turned
out to generalize best.

In [2]:
# Train the SVD++ model on the visible data and measure the training time. 
# We use the Surprise library's SVDpp implementation, which is an extension of the SVD algorithm that incorporates implicit feedback. 
# The model is trained with a specified number of latent factors and epochs, and we set a random seed for reproducibility.
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(visible[["userId", "movieId", "rating"]], reader)

model = SVDpp(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
model.fit(data.build_full_trainset())
print("Model trained.")

Model trained.


## Pick a user to trace

We specifically want someone who has ratings in BOTH visible and
hidden, so we can show all three views. Change `SAMPLE_USER` to any
userId you want to inspect — if they don't qualify (e.g. all their
ratings landed in one pile), the cell will tell you and suggest
another.

In [ ]:

SAMPLE_USER = 1   # change this to inspect a different user

visible_users = set(visible["userId"])
hidden_users  = set(hidden["userId"])
users_in_both = visible_users & hidden_users

if SAMPLE_USER not in users_in_both:
    fallback = sorted(users_in_both)[0]
    print(f"User {SAMPLE_USER} doesn't have ratings in both visible AND hidden.")
    print(f"Using user {fallback} instead (or edit SAMPLE_USER and re-run).")
    SAMPLE_USER = fallback
else:
    print(f"Tracing user {SAMPLE_USER}")

Tracing user 1


## View 1 — their full rating history (before any split)

In [ ]:
# Display the full rating history for the sample user, sorted by movieId. 
# This allows us to see all the ratings that the user has given, which can be useful for understanding their preferences and how the model might predict their ratings for unseen movies.

full_history = train[train["userId"] == SAMPLE_USER].sort_values("movieId")
print(f"Full history: {len(full_history)} ratings")
full_history[["userId", "movieId", "rating"]]

Full history: 28 ratings


,userId,movieId,rating
586417,1,296,5.0
235662,1,665,5.0
379417,1,899,3.5
851366,1,1175,3.5
66666,1,1217,3.5
523807,1,1237,5.0
180833,1,1250,4.0
266112,1,2068,2.5
860990,1,2351,4.5
139021,1,2632,5.0


## View 2 — what the model was allowed to see (visible)

In [5]:
their_visible = visible[visible["userId"] == SAMPLE_USER].sort_values("movieId")
print(f"Visible: {len(their_visible)} ratings (the model learned this user's taste from these)")
their_visible[["userId", "movieId", "rating"]]

Visible: 18 ratings (the model learned this user's taste from these)


,userId,movieId,rating
235662,1,665,5.0
379417,1,899,3.5
851366,1,1175,3.5
523807,1,1237,5.0
180833,1,1250,4.0
266112,1,2068,2.5
139021,1,2632,5.0
566956,1,2843,4.5
261162,1,4973,4.5
643553,1,6016,5.0


## View 3 — what was hidden from the model

The model has NEVER seen this user rate these specific movies.

In [ ]:
# Display the hidden ratings for the sample user, sorted by movieId.
# These are the ratings that the model has not seen during training, and we will use them to evaluate the model's performance in predicting the user's preferences.
their_hidden = hidden[hidden["userId"] == SAMPLE_USER].sort_values("movieId")
print(f"Hidden: {len(their_hidden)} ratings (model never saw these)")

# Sanity check: visible + hidden should equal the full history
assert len(their_visible) + len(their_hidden) == len(full_history), "Split doesn't add up!"
print(f"Check: visible({len(their_visible)}) + hidden({len(their_hidden)}) "
      f"== full history({len(full_history)}) -- confirmed")

their_hidden[["userId", "movieId", "rating"]]

Hidden: 10 ratings (model never saw these)
Check: visible(18) + hidden(10) == full history(28) -- confirmed


,userId,movieId,rating
586417,1,296,5.0
66666,1,1217,3.5
860990,1,2351,4.5
712551,1,3448,4.0
476202,1,3949,5.0
234828,1,4308,3.0
925253,1,5912,3.0
18056,1,6954,3.5
67210,1,7323,3.5
292173,1,27721,3.0


## Now the real test: predict their hidden ratings, compare to truth

For every movie this user rated that was held back, ask the model to
predict a rating — using ONLY what it learned from their visible
ratings (plus everyone else's patterns) — then compare to what they
actually rated.

In [ ]:
# Predict the ratings for the hidden set using the trained SVD++ model.
# We create a copy of the hidden set and use the model to predict the ratings for each movieId in the hidden set for the sample user.
their_hidden = their_hidden.copy()
their_hidden["predicted"] = [
    model.predict(SAMPLE_USER, m).est for m in their_hidden["movieId"]
]
their_hidden["predicted"] = their_hidden["predicted"].clip(0.5, 5.0)
their_hidden["abs_error"] = (their_hidden["rating"] - their_hidden["predicted"]).abs() # absolute error

their_hidden[["movieId", "rating", "predicted", "abs_error"]].round(2)

,movieId,rating,predicted,abs_error
586417,296,5.0,4.58,0.42
66666,1217,3.5,4.35,0.85
860990,2351,4.5,4.28,0.22
712551,3448,4.0,3.98,0.02
476202,3949,5.0,4.22,0.78
234828,4308,3.0,3.73,0.73
925253,5912,3.0,3.81,0.81
18056,6954,3.5,4.13,0.63
67210,7323,3.5,4.16,0.66
292173,27721,3.0,3.90,0.90


In [8]:
mae = their_hidden["abs_error"].mean()
print(f"This user's mean absolute error on hidden ratings: {mae:.3f} stars")
print(f"(For reference, the model's overall RMSE on ALL hidden ratings was ~0.86)")

best_pred = their_hidden.loc[their_hidden["abs_error"].idxmin()]
worst_pred = their_hidden.loc[their_hidden["abs_error"].idxmax()]
print(f"\nClosest prediction: movie {int(best_pred['movieId'])} "
      f"(actual {best_pred['rating']}, predicted {best_pred['predicted']:.2f})")
print(f"Furthest off: movie {int(worst_pred['movieId'])} "
      f"(actual {worst_pred['rating']}, predicted {worst_pred['predicted']:.2f})")

This user's mean absolute error on hidden ratings: 0.602 stars
(For reference, the model's overall RMSE on ALL hidden ratings was ~0.86)

Closest prediction: movie 3448 (actual 4.0, predicted 3.98)
Furthest off: movie 27721 (actual 3.0, predicted 3.90)


## What to take away from this

One user's average error won't exactly match the overall RMSE — that's
expected. RMSE is computed across tens of thousands of hidden ratings;
any single user is a small, somewhat noisy sample. Some users will be
predicted very accurately, others less so, depending on how much
visible data the model had about them and how typical or unusual their
taste is.

What this DOES confirm: the model is making genuinely personalized
predictions per movie — not just returning one flat number for this
user. That's the basic sanity check this exercise is meant to provide.

By changing `SAMPLE_USER` above to a few different values and
re-running the trace — you see the same mechanism applied to
different people, with the typical pattern of "mostly close, with
the occasional bigger miss" repeating.

## Next: predicting on the real test set

Everything so far has been rehearsal — using `train.csv`'s own hidden
ratings as a stand-in for the unknown. The next notebook does the real
version: retrain on ALL of `train.csv` (no holding back), then predict
every (user, movie) pair in `test.csv`, and write `submission.csv` in
the exact format Kaggle expects.